# Check `player_stats_39_2012.csv`

Run the cells top to bottom. The display settings below stop pandas from hiding columns, and the table views use scroll bars so you can inspect all 702 rows without losing the notebook layout.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)
pd.set_option("display.max_colwidth", 80)

LEAGUE_ID = 39
SEASON = 2012
CSV_PATH = Path("data") / "api_football" / "per_season" / "player_stats" / f"league_{LEAGUE_ID}" / f"player_stats_{LEAGUE_ID}_{SEASON}.csv"

df = pd.read_csv(CSV_PATH)
print(f"Loaded {CSV_PATH}")
print(f"Rows: {len(df):,} | Columns: {len(df.columns):,}")

In [ ]:
column_info = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null": df.notna().sum().values,
    "nulls": df.isna().sum().values,
    "null_pct": (df.isna().mean() * 100).round(1).values,
    "unique": df.nunique(dropna=True).values,
})

display(column_info)

In [ ]:
checks = {
    "rows": len(df),
    "columns": len(df.columns),
    "unique_players": df["player_id"].nunique() if "player_id" in df else None,
    "unique_teams": df["team_id"].nunique() if "team_id" in df else None,
    "duplicate_player_ids": df.duplicated("player_id").sum() if "player_id" in df else None,
    "rows_outside_league_season": len(df[(df["league_id"] != LEAGUE_ID) | (df["season"] != SEASON)])
    if {"league_id", "season"}.issubset(df.columns) else None,
}

display(pd.Series(checks, name="value").to_frame())

mostly_empty = column_info[column_info["null_pct"] >= 95].sort_values("null_pct", ascending=False)
print("Columns that are 95%+ empty:")
display(mostly_empty)

In [ ]:
def scroll_table(data, height=500):
    """Show a dataframe with vertical and horizontal scrolling."""
    html = data.to_html(index=True, max_rows=None, max_cols=None)
    return HTML(
        f'<div style="height:{height}px; overflow:auto; border:1px solid #ddd; '
        f'padding:8px; font-size:12px;">{html}</div>'
    )

scroll_table(df, height=650)

In [ ]:
important_cols = [
    "player_id", "player_name", "age", "nationality", "height", "weight",
    "team_name", "appearances", "lineups", "minutes", "position",
    "goals_total", "goals_assists", "cards_yellow", "cards_red",
]
important_cols = [c for c in important_cols if c in df.columns]

view = df[important_cols].sort_values(["team_name", "minutes", "player_name"], ascending=[True, False, True])
scroll_table(view, height=650)

In [ ]:
# Change these values when you want to search/filter.
PLAYER_CONTAINS = ""
TEAM_CONTAINS = ""
MIN_MINUTES = 0

filtered = df.copy()
if PLAYER_CONTAINS:
    filtered = filtered[filtered["player_name"].str.contains(PLAYER_CONTAINS, case=False, na=False)]
if TEAM_CONTAINS:
    filtered = filtered[filtered["team_name"].str.contains(TEAM_CONTAINS, case=False, na=False)]
if "minutes" in filtered:
    filtered = filtered[filtered["minutes"].fillna(0) >= MIN_MINUTES]

print(f"Matched rows: {len(filtered):,}")
scroll_table(filtered, height=500)

In [ ]:
if "player_id" in df.columns:
    duplicate_players = df[df.duplicated("player_id", keep=False)].sort_values(["player_id", "team_name"])
    print(f"Rows where the same player_id appears more than once: {len(duplicate_players):,}")
    display(duplicate_players[[c for c in important_cols if c in duplicate_players.columns]])
else:
    print("No player_id column found.")

In [ ]:
team_summary = (
    df.groupby("team_name", dropna=False)
    .agg(
        players=("player_id", "nunique"),
        rows=("player_id", "size"),
        total_minutes=("minutes", "sum"),
        total_goals=("goals_total", "sum"),
        total_yellows=("cards_yellow", "sum"),
        total_reds=("cards_red", "sum"),
    )
    .sort_values("team_name")
)

display(team_summary)

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns
display(df[numeric_cols].describe().T.round(2))